# BIST Altin Bazli ATL / ATH Tarayici v4

Her hisse icin: `Hisse (TL) / Altin (TL)` serisinin ATL ve ATH degerlerine uzaklik hesaplanir.

**v4:**
- Hisse listesi her calistirmada canli cekilir: TradingView BIST tarayicisi -> Is Yatirim -> statik yedek (625 hisse, Ekim 2026)
- Borsadan cikmis / kodu degismis eski semboller (KOZAL, MIGROS, QNBFB, VAKFB ...) temizlendi, eksik ~200 hisse eklendi
- Veriler toplu (batch) indirilir: tarama cok daha hizli
- Hatali Yahoo verisi (sifir / negatif fiyat) filtrelenir

**Calistir:** Calisma zamani -> Tumunu calistir

In [ ]:
#@title Ayarlar
ESIK_YUZDE = 30  #@param {type:"slider", min:5, max:100, step:5}
#@markdown Radar esigi (%): ATL'ye bu kadar yakin olanlar isaretlenir
MIN_VERI_GUNU = 30  #@param {type:"number"}
BATCH = 50  #@param {type:"number"}
#@markdown Tek seferde indirilecek hisse sayisi

In [ ]:
#@title Kurulum
!pip install -U yfinance pandas requests -q
import yfinance as yf, pandas as pd, requests, time, re
from datetime import datetime
from IPython.display import display
print('Kurulum tamam | yfinance', yf.__version__)

In [ ]:
#@title Guncel BIST hisse listesini cek

UA = {'User-Agent': 'Mozilla/5.0'}
# Hisse olmayan / taramaya girmemesi gerekenler (ALTIN = Darphane altin sertifikasi)
HARIC = {'ALTIN'}

def tradingview_listesi():
    """TradingView BIST tarayicisindan islem goren tum hisse senetleri."""
    payload = {
        'filter': [{'left': 'type', 'operation': 'equal', 'right': 'stock'},
                   {'left': 'exchange', 'operation': 'equal', 'right': 'BIST'}],
        'columns': ['name'],
        'range': [0, 3000],
    }
    r = requests.post('https://scanner.tradingview.com/turkey/scan',
                      json=payload, headers=UA, timeout=20)
    r.raise_for_status()
    return {x['d'][0].upper() for x in r.json()['data']
            if re.fullmatch(r'[A-Z0-9]{3,6}', x['d'][0].upper())}

def isyatirim_listesi():
    """Is Yatirim temel degerler sayfasindaki hisse kodlari."""
    url = 'https://www.isyatirim.com.tr/tr-tr/analiz/hisse/Sayfalar/Temel-Degerler-Ve-Oranlar.aspx'
    r = requests.get(url, headers=UA, timeout=30)
    r.raise_for_status()
    return set(re.findall(r'hisse=([A-Z0-9]{3,6})', r.text))

# Statik yedek liste (TradingView'den, Ekim 2026 - 625 hisse)
STATIK = set("""
A1CAP A1YEN AAGYO ACSEL ADEL ADESE ADGYO AEFES AFYON AGESA AGHOL AGROT AGYO AHGAZ AHSGY AKBNK
AKCNS AKENR AKFGY AKFIS AKFYE AKGRT AKHAN AKMGY AKSA AKSEN AKSGY AKSUE AKYHO ALARK ALBRK ALBTN
ALCAR ALCTL ALFAS ALGYO ALKA ALKIM ALKLC ALTNY ALVES ANELE ANGEN ANHYT ANSGR ARASE ARCLK ARDYZ
ARENA ARFYE ARMGD ARSAN ARTMS ARZUM ASELS ASGYO ASTOR ASUZU ATAGY ATAKP ATATP ATATR ATEKS ATLAS
ATSYH AVGYO AVHOL AVOD AVPGY AYCES AYDEM AYEN AYES AYGAZ AZTEK BAGFS BAHKM BAKAB BALAT BALSU
BANVT BARMA BASCM BASGZ BAYRK BEGYO BERA BESLR BESTE BETAE BEYAZ BFREN BIENY BIGCH BIGEN BIGTK
BIMAS BINBN BINHO BIOEN BIZIM BJKAS BKRGY BLCYT BLUME BMSCH BMSTL BNTAS BOBET BORLS BORSK BOSSA
BRISA BRKO BRKSN BRKVY BRLSM BRMEN BRSAN BRYAT BSOKE BTCIM BUCIM BULGS BURCE BURVA BVSAN BYDNR
CANTE CASA CATES CCOLA CELHA CEMAS CEMTS CEMZY CEOEM CGCAM CIMSA CITAS CLEBI CMBTN CMENT CONSE
COSMO CRDFA CRFSA CUSAN CVKMD CWENE DAGI DAPGM DARDL DCTTR DENGE DERHL DERIM DESA DESPC DEVA
DGATE DGGYO DGNMO DIRIT DITAS DMLKT DMRGD DMSAS DNISI DOAS DOCO DOFER DOFRB DOGUB DOHOL DOKTA
DSTKF DUNYH DURDO DURKN DYOBY DZGYO EBEBK ECILC ECOGR ECZYT EDATA EDIP EFOR EGEEN EGEGY EGEPO
EGGUB EGPRO EGSER EKDMR EKGYO EKIM EKIZ EKOS EKSUN ELITE EMKEL EMNIS EMPAE ENDAE ENERY ENJSA
ENKAI ENPRA ENSRI ENTRA EPLAS ERBOS ERCB EREGL ERSU ESCAR ESCOM ESEN ETILR ETYAT EUHOL EUKYO
EUPWR EUREN EUYO EYGYO FADE FENER FLAP FMIZP FONET FORMT FORTE FRIGO FRMPL FROTO FZLGY GARAN
GARFA GATEG GEDIK GEDZA GENIL GENKM GENTS GEREL GESAN GIPTA GLBMD GLCVY GLRMK GLRYH GLYHO GMTAS
GOKNR GOLDA GOLTS GOODY GOZDE GRNYO GRSEL GRTHO GSDDE GSDHO GSRAY GUBRF GUNDG GWIND GZNMI HALKB
HATEK HATSN HDFGS HEDEF HEKTS HKTM HLGYO HOROZ HRKET HTTBT HUBVC HUNER HURGZ ICBCT ICUGS IDGYO
IEYHO IHAAS IHEVA IHGZT IHLAS IHLGM IHYAY IMASM INDES INFO INGRM INTEK INTEM INTET INVEO INVES
ISBIR ISBTR ISCTR ISDMR ISFIN ISGYO ISKPL ISMEN ISSEN ISVEA IZENR IZFAS IZINV IZMDC JANTS KAPLM
KARCL KAREL KARSN KARTN KATMR KAYSE KBORU KCAER KCHOL KENT KERVN KFEIN KGYO KIMMR KLGYO KLKIM
KLMSN KLNMA KLRHO KLSER KLSYN KLYPV KMPUR KNFRT KOCMT KONKA KONTR KONYA KOPOL KORDS KOTON KPEKS
KRDMA KRDMB KRDMD KRGYO KRONT KRPLS KRSTL KRTEK KRVGD KSTUR KTLEV KTSKR KUTPO KUVVA KUYAS KZBGY
KZGYO LIDER LIDFA LILAK LINK LKMNH LMKDC LOGO LRSHO LUKSK LXGYO LYDHO LYDYE MAALT MACKO MAGEN
MAKIM MAKTK MANAS MARBL MARMR MARTI MASFN MAVI MCARD MEDTR MEGAP MEGMT MEKAG MEPET MERCN MERIT
MERKO METEN METRO MEYSU MGROS MHRGY MIATK MMCAS MNDRS MNDTR MOBTL MOGAN MOPAS MPARK MRGYO MRSHL
MSGYO MTRKS MTRYO MZHLD NATEN NETAS NETCD NETGL NIBAS NTGAZ NTHOL NUGYO NUHCM OBAMS OBASE ODAS
ODINE OFSYM ONCSM ONRYT ORCAY ORGE ORMA ORZAX OSMEN OSTIM OTKAR OTTO OYAKC OYAYO OYLUM OYYAT
OZATD OZGYO OZKGY OZRDN OZSUB OZYSR PAGYO PAHOL PAMEL PAPIL PARSN PASEU PATEK PCILT PEKGY PENGD
PENTA PETKM PETUN PGSUS PINSU PKART PKENT PLTUR PNLSN PNSUT POLHO POLTK PRDGS PRKAB PRKME PRZMA
PSDTC PSGYO QNBFK QNBTR QUAGR QUICK RALYH RAYSG REEDR RGYAS RNPOL RODRG RTALB RUBNS RUZYE RYGYO
RYSAS SAFKR SAHOL SAMAT SANEL SANFM SANKO SARAE SARKY SASA SAYAS SDTTR SEGMN SEGYO SEKFK SEKUR
SELEC SELVA SERNT SEYKM SILVR SISE SKBNK SKTAS SKYLP SKYMD SMART SMRTG SMRVA SNGYO SNICA SNPAM
SODSN SOHOE SOKE SOKM SONME SRVGY SSAAT SUMAS SUNTK SURGY SUWEN SVGYO TABGD TARKM TATEN TATGD
TAVHL TBORG TCELL TCKRC TDGYO TEHOL TEKTU TERA TEZOL TGSAS THYAO TKFEN TKNKA TKNSA TLMAN TMPOL
TMSN TNZTP TOASO TRALT TRCAS TRENJ TRGYO TRHOL TRILC TRMET TSGYO TSKB TSPOR TTKOM TTRAK TUCLK
TUKAS TUPRS TUREX TURGG TURSG UCAYM UFUK ULAS ULKER ULUFA ULUSE ULUUN UNLU USAK USHOL VAKBN
VAKFA VAKFN VAKKO VANGD VBTYZ VERUS VESBE VESTL VEYAS VKFYO VKGYO VKING VRGYO VSNMD YAPRK YATAS
YAYLA YBTAS YEOTK YESIL YGGYO YIGIT YKBNK YKSLN YONGA YUNSA YYAPI YYLGD ZEDUR ZERGY ZGYO ZOREN
ZRGYO
""".split())

def liste_olustur():
    for ad, fn in [('TradingView', tradingview_listesi), ('Is Yatirim', isyatirim_listesi)]:
        try:
            s = fn()
            print(f'  {ad}: {len(s)} hisse')
            if len(s) >= 400:
                return s, ad
            print(f'  {ad} listesi eksik gorunuyor, sonraki kaynaga geciliyor')
        except Exception as e:
            print(f'  {ad} hatasi: {e}')
    print('  Canli kaynaklara ulasilamadi, statik liste kullaniliyor')
    return set(STATIK), 'Statik'

print('Hisse listesi olusturuluyor...')
canli, kaynak = liste_olustur()
tum_tikerlar = sorted(canli - HARIC)

yeni   = sorted(set(tum_tikerlar) - STATIK)
cikmis = sorted(STATIK - set(tum_tikerlar) - HARIC)
print(f'\nKaynak: {kaynak} | Toplam: {len(tum_tikerlar)} hisse')
if kaynak != 'Statik':
    if yeni:   print(f'Statik listede olmayan yeni hisseler ({len(yeni)}):', ', '.join(yeni))
    if cikmis: print(f'Artik listede olmayanlar ({len(cikmis)}):', ', '.join(cikmis))

In [ ]:
#@title Altin/TL paritesi (XAUUSD x USDTRY)

def tek_seri(sembol, deneme=3):
    for _ in range(deneme):
        try:
            df = yf.download(sembol, period='max', interval='1d', auto_adjust=True,
                             progress=False, multi_level_index=False)
            if not df.empty and 'Close' in df.columns:
                return df['Close'].dropna()
        except Exception:
            pass
        time.sleep(2)
    raise RuntimeError(f'{sembol} verisi alinamadi')

xauusd = tek_seri('GC=F')
usdtry = tek_seri('USDTRY=X')
ix = xauusd.index.intersection(usdtry.index)
xautry = (xauusd.loc[ix] * usdtry.loc[ix]).dropna()
xautry = xautry[xautry > 0]
print(f'Sentetik XAUTRY: {xautry.index[0]:%Y-%m-%d} - {xautry.index[-1]:%Y-%m-%d} ({len(xautry)} gun)')

In [ ]:
#@title Tarama (ATL + ATH)

def toplu_indir(tikerlar):
    """Hisseleri BATCH'ler halinde indirir; {ticker: Close serisi} dondurur."""
    veriler = {}
    for b in range(0, len(tikerlar), BATCH):
        grup = tikerlar[b:b+BATCH]
        semboller = [f'{t}.IS' for t in grup]
        for deneme in range(3):
            try:
                df = yf.download(semboller, period='max', interval='1d', auto_adjust=True,
                                 progress=False, group_by='ticker', threads=True)
                break
            except Exception as e:
                print(f'  batch hatasi ({e}), tekrar deneniyor...'); time.sleep(3)
        else:
            continue
        for t, s in zip(grup, semboller):
            try:
                c = df[s]['Close'] if isinstance(df.columns, pd.MultiIndex) else df['Close']
                c = c.dropna()
                if len(c): veriler[t] = c
            except KeyError:
                pass
        print(f'  {min(b+BATCH, len(tikerlar))}/{len(tikerlar)} indirildi ({len(veriler)} veri var)')
    return veriler

print(f'{len(tum_tikerlar)} hisse indiriliyor...')
veriler = toplu_indir(tum_tikerlar)

sonuclar, hatalar = [], {}
for ticker in tum_tikerlar:
    h = veriler.get(ticker)
    if h is None:
        hatalar[ticker] = 'veri yok'; continue
    h = h[h > 0]                         # hatali (0 / negatif) fiyatlari at
    ort = h.index.intersection(xautry.index)
    if len(ort) < MIN_VERI_GUNU:
        hatalar[ticker] = f'az veri ({len(ort)} gun)'; continue
    seri = (h.loc[ort] / xautry.loc[ort]).dropna()

    atl, ath, son = float(seri.min()), float(seri.max()), float(seri.iloc[-1])
    sonuclar.append({
        'Hisse'       : ticker,
        'Son Tarih'   : f'{seri.index[-1]:%Y-%m-%d}',
        'Son (XAU)'   : son,
        'ATL (XAU)'   : atl,
        'ATL Tarihi'  : f'{seri.idxmin():%Y-%m-%d}',
        'ATL Uzaklik%': round((son - atl) / atl * 100, 2),   # ATL'den ne kadar yukarida
        'ATH (XAU)'   : ath,
        'ATH Tarihi'  : f'{seri.idxmax():%Y-%m-%d}',
        'ATH Uzaklik%': round((son - ath) / ath * 100, 2),   # ATH'nin ne kadar altinda
        'Gun'         : len(seri),
    })

print(f'\n{len(sonuclar)} hisse islendi | {len(hatalar)} atlandi')
if hatalar:
    print('Atlananlar:', ', '.join(f'{k} ({v})' for k, v in hatalar.items()))

In [ ]:
#@title Sonuc tablolari
df = pd.DataFrame(sonuclar).sort_values('ATL Uzaklik%').reset_index(drop=True)
pd.set_option('display.float_format', lambda x: f'{x:.8f}' if 0 < abs(x) < 1 else f'{x:.2f}')
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 700)
pd.set_option('display.width', 200)

# Son verisi 7 gunden eski olanlar (islem durdurma vb.) uyari icin isaretlenir
son_gun = pd.to_datetime(df['Son Tarih']).max()
eski = df[pd.to_datetime(df['Son Tarih']) < son_gun - pd.Timedelta(days=7)]
if len(eski):
    print(f'Uyari - son verisi eski olan {len(eski)} hisse:', ', '.join(eski['Hisse']))

radar_df = df[df['ATL Uzaklik%'] <= ESIK_YUZDE].reset_index(drop=True)
print(f'\nRADAR - ATL uzaklik %{ESIK_YUZDE} veya daha az: {len(radar_df)} hisse')
print('='*80)
display(radar_df)
print('\nTUM LISTE (ATL yakinlik sirasi):')
print('='*80)
display(df)

In [ ]:
#@title CSV indir
from google.colab import files
dosya = f'bist_atl_ath_{datetime.now():%Y%m%d_%H%M}.csv'
df.to_csv(dosya, index=False, encoding='utf-8-sig', float_format='%.8f')
files.download(dosya)
print(f'{dosya} indirildi')